# Плановый ФОТ в Jupyter: от метода ЗУП к hot reload

Выберем всех работающих сотрудников, получим плановый ФОТ типовым методом, посмотрим результат в Python и добавим страховые взносы через hot reload.

Пример проверен с выгрузкой ЗУП КОРП 3.1.38.92 и платформой 8.5.1.1529; дата демоснимка — 01.08.2021. Используйте отдельную демо-копию и соответствующую ей выгрузку исходников. Подготовка окружения — в [README](README.md). Сеанс закрывается последней ячейкой; при прерывании выполните `runtime.close()`.

## Подготовка

Этот вариант использует API wheels 0.1.17. Перед запуском измените `PLATFORM_BIN`, `CONNECTION_STRING` и `SOURCE_ROOT` в следующей ячейке. Имя пользователя уже задано; пароль пустой.

In [ ]:
from IPython.display import display
from onec_runtime.config import RuntimeConfig
from onec_runtime.session import ExtensionMode, RuntimeSessionConfig
from onec_runtime_jupyter import InteractiveRuntimeSession

PLATFORM_BIN = r'C:\Program Files\1cv8\8.5.1.1529\bin'
CONNECTION_STRING = r'File="C:\demo\ЗУП";'
SOURCE_ROOT = r'C:\exports\ЗУП'
EXTENSION_MODE = ExtensionMode.AUTO  # MANUAL для ИБ с заранее установленным расширением.

runtime = InteractiveRuntimeSession.start(
    RuntimeSessionConfig(
        runtime=RuntimeConfig(
            platform_bin=PLATFORM_BIN,
            connection_string=CONNECTION_STRING,
            username='Савинская З.Ю. (Системный программист)',
        ),
        source_root=SOURCE_ROOT,
        extension_mode=EXTENSION_MODE,
    )
)

## Выбираем сотрудников

Типовой метод КадровыйУчет.СотрудникиОрганизации возвращает работающих по трудовым договорам на 01.08.2021. Для метода оплаты труда оставляем уникальные пары Сотрудник и Период.

In [ ]:
%%bsl
ДатаФОТ = Дата(2021, 8, 1);
ПараметрыСотрудников = КадровыйУчет.ПараметрыПолученияСотрудниковОрганизацийПоСпискуФизическихЛиц();
ПараметрыСотрудников.НачалоПериода = ДатаФОТ;
ПараметрыСотрудников.ОкончаниеПериода = ДатаФОТ;
ПараметрыСотрудников.РаботникиПоТрудовымДоговорам = Истина;

СписокСотрудников = КадровыйУчет.СотрудникиОрганизации(
    Истина, ПараметрыСотрудников).Скопировать(, "Сотрудник");
СписокСотрудников.Свернуть("Сотрудник");
СписокСотрудников.Колонки.Добавить("Период", Новый ОписаниеТипов("Дата"));
Для Каждого СтрокаСотрудника Из СписокСотрудников Цикл
    СтрокаСотрудника.Период = ДатаФОТ;
КонецЦикла;

## Текущие данные оплаты труда

Типовой метод возвращает ФОТ, период и тарифные показатели. Для диаграммы берём только Сотрудник и ФОТ. Отсутствующие в регистре сотрудники могут не попасть в результат.

In [ ]:
%%bsl
ПлановыйФот = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, СписокСотрудников);
ПланФОТ = ПлановыйФот.Скопировать(, "Сотрудник,ФОТ");

## Таблица и диаграмма

Имена BSL-переменных доступны в Python без префикса. Представление ссылки подходит для подписи столбцов; это снимок, который не меняется после hot reload.

In [ ]:
# Переменные из %%bsl доступны в Python по тем же именам.
plan = ПланФОТ.to_df(refs='presentation')
display(plan)
print('Сотрудников с данными:', len(plan))
print('Плановый ФОТ:', plan['ФОТ'].sum(), '₽')

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, max(5, len(plan) * 0.32)))
ax.barh(plan['Сотрудник'], plan['ФОТ'].map(float))
ax.invert_yaxis()
ax.set_xlabel('Плановый ФОТ, ₽')
ax.set_title('Плановый ФОТ работающих сотрудников на 01.08.2021')
fig.tight_layout()
display(fig)
plt.close(fig)

## Внутри типового метода

Точка на строке возврата позволяет посмотреть локальную таблицу до завершения вызова. Путь может быть абсолютным или относительным к SOURCE_ROOT. Перед запуском укажите в CAPTURE_LINE строку Возврат ЗначенияДанныхОплатыТруда; из своей выгрузки.

In [ ]:
MODULE_PATH = r'CommonModules\ПлановыеНачисленияСотрудников\Ext\Module.bsl'
CAPTURE_LINE = 745  # Строка Возврат ЗначенияДанныхОплатыТруда; в вашей выгрузке.
runtime.add_capture_point(MODULE_PATH, CAPTURE_LINE)

In [ ]:
%%bsl
ПланПовтор = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, СписокСотрудников);

In [ ]:
%%bsl
СнимокВызова = КонтекстОтладки.ЗначенияДанныхОплатыТруда.Скопировать(, "Сотрудник,ФОТ");

In [ ]:
captured_plan = СнимокВызова.to_df(refs='presentation')
display(captured_plan)

In [ ]:
completed = runtime.resume_capture()
runtime.clear_capture_points()
print('Состояние вызова:', completed.state.value)

In [ ]:
%%bsl
ПланПовторКратко = ПланПовтор.Скопировать(, "Сотрудник,ФОТ");

In [ ]:
resumed_plan = ПланПовторКратко.to_df(refs='presentation')
display(resumed_plan)

## Hot reload: ФОТ со страховыми взносами

В выгрузке SOURCE_ROOT откройте CommonModules/ПлановыеНачисленияСотрудников/Ext/Module.bsl. Замените метод ТекущиеДанныеОплатыТрудаСотрудников кодом ниже и сохраните файл. ФОТСоСтраховыми содержит ФОТ с условными страховыми взносами 30%:

```bsl
Функция ТекущиеДанныеОплатыТрудаСотрудников(Ссылка, СотрудникиДаты) Экспорт
    ПараметрыПостроения = ЗарплатаКадрыОбщиеНаборыДанных.ПараметрыПостроенияДляСоздатьВТИмяРегистраСрез();
    ПараметрыПостроения.ФормироватьСПериодичностьДень = Ложь;
    ЗарплатаКадрыОбщиеНаборыДанных.ДобавитьВКоллекциюОтбор(
        ПараметрыПостроения.Отборы, "Регистратор", "<>", Ссылка);
    Запрос = Новый Запрос;
    Запрос.МенеджерВременныхТаблиц = Новый МенеджерВременныхТаблиц;
    ЗарплатаКадрыОбщиеНаборыДанных.СоздатьВТИмяРегистраСрезПоследних(
        "ПлановыйФОТИтоги",
        Запрос.МенеджерВременныхТаблиц,
        Истина,
        ЗарплатаКадрыОбщиеНаборыДанных.ОписаниеФильтраДляСоздатьВТИмяРегистра(
            СотрудникиДаты, "Сотрудник"),
        ПараметрыПостроения);
    Запрос.Текст =
        "ВЫБРАТЬ
        |   ЗначенияСовокупныхТарифныхСтавок.Сотрудник,
        |   ЗначенияСовокупныхТарифныхСтавок.Период,
        |   ЗначенияСовокупныхТарифныхСтавок.СовокупнаяТарифнаяСтавка КАК СовокупнаяТарифнаяСтавка,
        |   ЗначенияСовокупныхТарифныхСтавок.ВидТарифнойСтавки,
        |   ЗначенияСовокупныхТарифныхСтавок.ФОТ КАК ФОТ,
        |   ЗначенияСовокупныхТарифныхСтавок.ФОТ * 1.30 КАК ФОТСоСтраховыми
        |ИЗ
        |   ВТПлановыйФОТИтогиСрезПоследних КАК ЗначенияСовокупныхТарифныхСтавок";
    ЗначенияДанныхОплатыТруда = Запрос.Выполнить().Выгрузить();
    Возврат ЗначенияДанныхОплатыТруда;
КонецФункции
```

Правка меняет файл выгрузки на диске, но не конфигурацию ИБ. Hot reload применяет сохранённый Module.bsl к этому runtime-сеансу.

In [ ]:
runtime.load_worker_module(MODULE_PATH);

In [ ]:
%%bsl
ПлановыйФотСоСтраховыми = ПлановыеНачисленияСотрудников.ТекущиеДанныеОплатыТрудаСотрудников(
    Неопределено, СписокСотрудников);
ПланСоСтраховыми = ПлановыйФотСоСтраховыми.Скопировать(, "Сотрудник,ФОТ,ФОТСоСтраховыми");

In [ ]:
reloaded = ПланСоСтраховыми.to_df(refs="presentation")
display(reloaded)
print("ФОТ:", reloaded["ФОТ"].sum(), "₽")
print("ФОТ со страховыми:", reloaded["ФОТСоСтраховыми"].sum(), "₽")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['ФОТ', 'ФОТ со страховыми'], [
    float(reloaded['ФОТ'].sum()),
    float(reloaded['ФОТСоСтраховыми'].sum()),
])
ax.set_ylabel('Сумма по выбранным сотрудникам, ₽')
ax.set_title('Результат перегруженного метода')
fig.tight_layout()
display(fig)
plt.close(fig)

## Дальше

В [03-capture.ipynb](03-capture.ipynb) показаны две точки останова, стек вызовов и изменение локального запроса.

## Завершение

In [ ]:
runtime.close()